In [1]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
3,application_1791649099190_0004,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
from datetime import datetime
from dateutil.relativedelta import relativedelta 

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ' >>>'

dt_proc = datetime.now().strftime('%Y%m%d%H%M%S')
current_date = datetime.now().strftime('%Y%m%d')


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [76]:
dt_proc_book = datetime.strptime('2024-09-30','%Y-%m-%d')
dt_proc_book_str = datetime.strptime('2024-09-30','%Y-%m-%d').strftime('%Y-%m-%d')
dt_ini_proc_book = dt_proc_book - relativedelta(months=12)

dt_proc_book = dt_proc_book.strftime('%Y%m')
dt_ini_proc_book = dt_ini_proc_book.strftime('%Y%m')


dt_proc_book, dt_ini_proc_book, dt_proc_book_str

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

('202409', '202309', '2024-09-30')

In [12]:
# carregar a tabela de fatruas


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [16]:
tb_fatura = spark.read.parquet('s3://data-lake-elt-303004020922-us-east-1-an/02_trusted/tb_0001_fatura/')
tb_fatura_00 = tb_fatura.where (f'ref >= {dt_ini_proc_book} and ref <= {dt_proc_book}')
tb_fatura_00.createOrReplaceTempView('tb_fatura_00')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [35]:
spark.sql('select min(ref),max(ref) from tb_fatura_00').show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+--------+
|min(ref)|max(ref)|
+--------+--------+
|  202309|  202409|
+--------+--------+

In [20]:
#verificando se exite registro duplicado

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
spark.sql("""
            select id_cliente,id_fatura,count(*) 
            from tb_fatura_00
            group by 1,2
            order by 3 desc
""").show(4)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+---------+--------+
|id_cliente|id_fatura|count(1)|
+----------+---------+--------+
|        93|     1452|       1|
|       884|    13770|       1|
|       970|    15074|       1|
|       366|     5689|       1|
+----------+---------+--------+
only showing top 4 rows

In [24]:
#carregar a tabela de pagamentos


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [25]:
tb_pagamentos = spark.read.parquet('s3://data-lake-elt-303004020922-us-east-1-an/02_trusted/tb_0002_pagamentos/')
tb_pagamentos_00 = tb_pagamentos.where (f'ref >= {dt_ini_proc_book} and ref <= {dt_proc_book}')
tb_pagamentos_00.createOrReplaceTempView('tb_pagamentos_00')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [34]:
spark.sql('select min(ref),max(ref) from tb_pagamentos_00').show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+--------+
|min(ref)|max(ref)|
+--------+--------+
|  202309|  202409|
+--------+--------+

In [32]:
spark.sql("""
            select id_cliente,id_fatura,count(*) 
            from tb_pagamentos_00
            group by 1,2
            order by 3 desc
""").show(4)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+---------+--------+
|id_cliente|id_fatura|count(1)|
+----------+---------+--------+
|       970|    15074|       1|
|       173|     2707|       1|
|       554|     8610|       1|
|       288|     4490|       1|
+----------+---------+--------+
only showing top 4 rows

In [28]:
#etapa de stage onde serão criadas as variáveis categoricas e as classificações do book


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [44]:
df_join = spark.sql("""
          select 
                a.id_cliente,
                a.id_fatura,
                a.dt_emissao_fatura,
                a.dt_vencimento_fatura,
                a.valor_fatura,
                b.id_pagamento,
                b.dt_pagamento,
                b.valor_pagamento
           from tb_fatura_00 a
           left join tb_pagamentos_00 b
           on a.id_fatura = b.id_fatura and a.id_cliente = b.id_cliente
           order by a.id_cliente,a.id_fatura
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [45]:
df_join.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+---------+-----------------+--------------------+------------+------------+------------+---------------+
|id_cliente|id_fatura|dt_emissao_fatura|dt_vencimento_fatura|valor_fatura|id_pagamento|dt_pagamento|valor_pagamento|
+----------+---------+-----------------+--------------------+------------+------------+------------+---------------+
|         1|        9|       2023-09-01|          2023-09-06|     4912.10|        NULL|        NULL|           NULL|
|         1|       10|       2023-10-01|          2023-10-06|     4983.78|           6|  2023-10-11|        4983.78|
|         1|       11|       2024-01-01|          2024-01-06|     3836.72|        NULL|        NULL|           NULL|
|         2|       20|       2023-09-01|          2023-09-06|     2590.41|        NULL|        NULL|           NULL|
|         2|       21|       2023-10-01|          2023-10-06|     2170.08|        NULL|        NULL|           NULL|
+----------+---------+-----------------+--------------------+---

In [54]:
df_join.createOrReplaceTempView('df_join')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [49]:
#criando as classificações

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [70]:
tb_classificacao_pagamentos= spark.sql("""
            select 
                *,
                case 
                    when dt_pagamento is null then 'SEM PAGAMENTO'
                    when dt_pagamento = dt_vencimento_fatura then 'PAGAMENTO NO PRAZO'
                    when dt_pagamento < dt_vencimento_fatura then 'PAGAMENTO NO ANTECIPADO'
                    when dt_pagamento > dt_vencimento_fatura then 'PAGAMENTO EM ATRASO'
                end fbc_classificacao_pagamento
                from df_join
""")



FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [72]:
tb_classificacao_pagamentos.createOrReplaceTempView('tb_classificacao_pagamentos')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
+----------+---------+-----------------+--------------------+------------+------------+------------+---------------+---------------------------+
|id_cliente|id_fatura|dt_emissao_fatura|dt_vencimento_fatura|valor_fatura|id_pagamento|dt_pagamento|valor_pagamento|fbc_classificacao_pagamento|
+----------+---------+-----------------+--------------------+------------+------------+------------+---------------+---------------------------+
|         1|        9|       2023-09-01|          2023-09-06|     4912.10|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         1|       10|       2023-10-01|          2023-10-06|     4983.78|           6|  2023-10-11|        4983.78|        PAGAMENTO EM ATRASO|
|         1|       11|       2024-01-01|          2024-01-06|     3836.72|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         2|       20|       2023-09-01|          2023-09-06|     2590.41|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         2|       21|       2023-10-01|          2023-10-06|     2170.08|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         2|       22|       2023-11-01|          2023-11-06|     3346.01|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         2|       23|       2024-04-01|          2024-04-06|     4964.46|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         2|       24|       2024-07-01|          2024-07-06|      550.87|           8|  2024-07-16|         550.87|        PAGAMENTO EM ATRASO|
|         2|       25|       2024-09-01|          2024-09-06|     1882.13|           9|  2024-09-20|        1882.13|        PAGAMENTO EM ATRASO|
|         3|       34|       2023-09-01|          2023-09-06|     1596.95|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         3|       35|       2023-10-01|          2023-10-06|       74.85|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         3|       36|       2024-02-01|          2024-02-06|     2893.24|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         3|       37|       2024-04-01|          2024-04-06|     4285.20|          13|  2024-04-16|        4285.20|        PAGAMENTO EM ATRASO|
|         3|       38|       2024-09-01|          2024-09-06|     1529.37|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         4|       47|       2023-09-01|          2023-09-06|     4190.98|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         4|       48|       2023-10-01|          2023-10-06|     2937.44|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         4|       49|       2024-01-01|          2024-01-06|     3030.63|          17|  2024-01-09|        3030.63|        PAGAMENTO EM ATRASO|
|         4|       50|       2024-02-01|          2024-02-06|     3380.91|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
|         4|       51|       2024-04-01|          2024-04-06|     4354.85|          18|  2024-04-06|        4354.85|         PAGAMENTO NO PRAZO|
|         4|       52|       2024-06-01|          2024-06-06|      125.80|        NULL|        NULL|           NULL|              SEM PAGAMENTO|
+----------+---------+-----------------+--------------------+------------+------------+------------+---------------+---------------------------+
only showing top 20 rows

In [74]:
#criando a variável qtd de dias em atraso


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [79]:
tb_dias_atraso= spark.sql("""
            select 
                *,
                case 
                    when fbc_classificacao_pagamento = 'SEM PAGAMENTO' then datediff("{dt_proc_book_str}",dt_vencimento_fatura)
                    when fbc_classificacao_pagamento = 'PAGAMENTO EM ATRASO' then datediff(dt_pagamento, dt_vencimento_fatura)
                    when fbc_classificacao_pagamento = 'PAGAMENTO NO PRAZO'  then 0
                    when fbc_classificacao_pagamento = 'PAGAMENTO NO ANTECIPADO' then -1
                end fvl_numero_dias_atraso
                from tb_classificacao_pagamentos
""")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [80]:
tb_dias_atraso.createOrReplaceTempView('tb_dias_atraso')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
+----------+---------+-----------------+--------------------+------------+------------+------------+---------------+---------------------------+----------------------+
|id_cliente|id_fatura|dt_emissao_fatura|dt_vencimento_fatura|valor_fatura|id_pagamento|dt_pagamento|valor_pagamento|fbc_classificacao_pagamento|fvl_numero_dias_atraso|
+----------+---------+-----------------+--------------------+------------+------------+------------+---------------+---------------------------+----------------------+
|         1|        9|       2023-09-01|          2023-09-06|     4912.10|        NULL|        NULL|           NULL|              SEM PAGAMENTO|                  NULL|
|         1|       10|       2023-10-01|          2023-10-06|     4983.78|           6|  2023-10-11|        4983.78|        PAGAMENTO EM ATRASO|                     5|
|         1|       11|       2024-01-01|          2024-01-06|     3836.72|        NULL|        NULL|           NULL|              SEM PAGAMENTO|                  NULL|
|         2|       20|       2023-09-01|          2023-09-06|     2590.41|        NULL|        NULL|           NULL|              SEM PAGAMENTO|                  NULL|
+----------+---------+-----------------+--------------------+------------+------------+------------+---------------+---------------------------+----------------------+

In [ ]:
#finalização da stage
# a stage é uma camada pré-book onde os dados são salvos já categorizados


In [111]:
stage = spark.sql(f"""
            select 
                id_cliente,
                substring(replace("{dt_proc_book_str}",'-',''),1,6) as ref,
                "{dt_proc}" as dt_proc,
                fbc_classificacao_pagamento,
                dt_emissao_fatura,
                valor_fatura as fvl_valor_fatura,
                fvl_numero_dias_atraso,
                1 as fvl_qtd_transacao
            from tb_dias_atraso               
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [112]:
stage.createOrReplaceTempView('Stage')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
+----------+------+--------------+---------------------------+-----------------+----------------+----------------------+-----------------+
|id_cliente|   ref|       dt_proc|fbc_classificacao_pagamento|dt_emissao_fatura|fvl_valor_fatura|fvl_numero_dias_atraso|fvl_qtd_transacao|
+----------+------+--------------+---------------------------+-----------------+----------------+----------------------+-----------------+
|         1|202409|20261010202441|              SEM PAGAMENTO|       2023-09-01|         4912.10|                  NULL|                1|
|         1|202409|20261010202441|        PAGAMENTO EM ATRASO|       2023-10-01|         4983.78|                     5|                1|
|         1|202409|20261010202441|              SEM PAGAMENTO|       2024-01-01|         3836.72|                  NULL|                1|
|         2|202409|20261010202441|              SEM PAGAMENTO|       2023-09-01|         2590.41|                  NULL|                1|
|         2|202409|20261010202441|              SEM PAGAMENTO|       2023-10-01|         2170.08|                  NULL|                1|
|         2|202409|20261010202441|              SEM PAGAMENTO|       2023-11-01|         3346.01|                  NULL|                1|
|         2|202409|20261010202441|              SEM PAGAMENTO|       2024-04-01|         4964.46|                  NULL|                1|
|         2|202409|20261010202441|        PAGAMENTO EM ATRASO|       2024-07-01|          550.87|                    10|                1|
|         2|202409|20261010202441|        PAGAMENTO EM ATRASO|       2024-09-01|         1882.13|                    14|                1|
|         3|202409|20261010202441|              SEM PAGAMENTO|       2023-09-01|         1596.95|                  NULL|                1|
|         3|202409|20261010202441|              SEM PAGAMENTO|       2023-10-01|           74.85|                  NULL|                1|
|         3|202409|20261010202441|              SEM PAGAMENTO|       2024-02-01|         2893.24|                  NULL|                1|
|         3|202409|20261010202441|        PAGAMENTO EM ATRASO|       2024-04-01|         4285.20|                    10|                1|
|         3|202409|20261010202441|              SEM PAGAMENTO|       2024-09-01|         1529.37|                  NULL|                1|
|         4|202409|20261010202441|              SEM PAGAMENTO|       2023-09-01|         4190.98|                  NULL|                1|
|         4|202409|20261010202441|              SEM PAGAMENTO|       2023-10-01|         2937.44|                  NULL|                1|
|         4|202409|20261010202441|        PAGAMENTO EM ATRASO|       2024-01-01|         3030.63|                     3|                1|
|         4|202409|20261010202441|              SEM PAGAMENTO|       2024-02-01|         3380.91|                  NULL|                1|
|         4|202409|20261010202441|         PAGAMENTO NO PRAZO|       2024-04-01|         4354.85|                     0|                1|
|         4|202409|20261010202441|              SEM PAGAMENTO|       2024-06-01|          125.80|                  NULL|                1|
+----------+------+--------------+---------------------------+-----------------+----------------+----------------------+-----------------+

In [114]:
stage.write.mode("append").partitionBy("ref").parquet("s3://data-lake-elt-303004020922-us-east-1-an/03_refined/stage/stage_fatura")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [110]:
#book de variáveis

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [129]:
janelas_tempo = spark.sql(f"""
select
    *,
    (case when months_between("{dt_proc_book_str}", dt_emissao_fatura) <= 1 then 1 else 0 end) flg_u1m,
    (case when months_between("{dt_proc_book_str}", dt_emissao_fatura) <= 3 then 1 else 0 end) flg_u3m,
    (case when months_between("{dt_proc_book_str}", dt_emissao_fatura) <= 6 then 1 else 0 end) flg_u6m,
    (case when months_between("{dt_proc_book_str}", dt_emissao_fatura) <= 9 then 1 else 0 end) flg_u9m,
    (case when months_between("{dt_proc_book_str}", dt_emissao_fatura) <= 12 then 1 else 0 end) flg_u12m
from stage
""")
janelas_tempo.createOrReplaceTempView('janelas_tempo')


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
+----------+------+--------------+---------------------------+-----------------+----------------+----------------------+-----------------+-------+-------+-------+-------+--------+
|id_cliente|   ref|       dt_proc|fbc_classificacao_pagamento|dt_emissao_fatura|fvl_valor_fatura|fvl_numero_dias_atraso|fvl_qtd_transacao|flg_u1m|flg_u3m|flg_u6m|flg_u9m|flg_u12m|
+----------+------+--------------+---------------------------+-----------------+----------------+----------------------+-----------------+-------+-------+-------+-------+--------+
|         1|202409|20261010202441|              SEM PAGAMENTO|       2023-09-01|         4912.10|                  NULL|                1|      0|      0|      0|      0|       0|
|         1|202409|20261010202441|        PAGAMENTO EM ATRASO|       2023-10-01|         4983.78|                     5|                1|      0|      0|      0|      0|       1|
|         1|202409|20261010202441|              SEM PAGAMENTO|       2024-01-01|         3836.72|                  NULL|                1|      0|      0|      0|      1|       1|
|         2|202409|20261010202441|              SEM PAGAMENTO|       2023-09-01|         2590.41|                  NULL|                1|      0|      0|      0|      0|       0|
|         2|202409|20261010202441|              SEM PAGAMENTO|       2023-10-01|         2170.08|                  NULL|                1|      0|      0|      0|      0|       1|
|         2|202409|20261010202441|              SEM PAGAMENTO|       2023-11-01|         3346.01|                  NULL|                1|      0|      0|      0|      0|       1|
|         2|202409|20261010202441|              SEM PAGAMENTO|       2024-04-01|         4964.46|                  NULL|                1|      0|      0|      1|      1|       1|
|         2|202409|20261010202441|        PAGAMENTO EM ATRASO|       2024-07-01|          550.87|                    10|                1|      0|      1|      1|      1|       1|
|         2|202409|20261010202441|        PAGAMENTO EM ATRASO|       2024-09-01|         1882.13|                    14|                1|      1|      1|      1|      1|       1|
|         3|202409|20261010202441|              SEM PAGAMENTO|       2023-09-01|         1596.95|                  NULL|                1|      0|      0|      0|      0|       0|
|         3|202409|20261010202441|              SEM PAGAMENTO|       2023-10-01|           74.85|                  NULL|                1|      0|      0|      0|      0|       1|
|         3|202409|20261010202441|              SEM PAGAMENTO|       2024-02-01|         2893.24|                  NULL|                1|      0|      0|      0|      1|       1|
|         3|202409|20261010202441|        PAGAMENTO EM ATRASO|       2024-04-01|         4285.20|                    10|                1|      0|      0|      1|      1|       1|
|         3|202409|20261010202441|              SEM PAGAMENTO|       2024-09-01|         1529.37|                  NULL|                1|      1|      1|      1|      1|       1|
|         4|202409|20261010202441|              SEM PAGAMENTO|       2023-09-01|         4190.98|                  NULL|                1|      0|      0|      0|      0|       0|
|         4|202409|20261010202441|              SEM PAGAMENTO|       2023-10-01|         2937.44|                  NULL|                1|      0|      0|      0|      0|       1|
|         4|202409|20261010202441|        PAGAMENTO EM ATRASO|       2024-01-01|         3030.63|                     3|                1|      0|      0|      0|      1|       1|
|         4|202409|20261010202441|              SEM PAGAMENTO|       2024-02-01|         3380.91|                  NULL|                1|      0|      0|      0|      1|       1|
|         4|202409|20261010202441|         PAGAMENTO NO PRAZO|       2024-04-01|         4354.85|                     0|                1|      0|      0|      1|      1|       1|
|         4|202409|20261010202441|              SEM PAGAMENTO|       2024-06-01|          125.80|                  NULL|                1|      0|      0|      1|      1|       1|
+----------+------+--------------+---------------------------+-----------------+----------------+----------------------+-----------------+-------+-------+-------+-------+--------+

In [ ]:
#meta programação de apoio para a construção das variáveis do book

In [149]:
fbcs = ['PAGAMENTO NO PRAZO', 'PAGAMENTO ANTECIPADO', 'SEM PAGAMENTO', 'PAGAMENTO EM ATRASO']
fvls = ['fvl_valor_fatura', 'fvl_qtd_transacao', 'fvl_numero_dias_atraso']
janelas = ['1', 'FLG_U1M', 'FLG_U3M', 'FLG_U6M', 'FLG_U9M', 'FLG_U12M']
aggs = ['SUM', 'AVG', 'MIN', 'MAX']

count = 1

for fbc in fbcs:
    for fvl in fvls:
        for janela in janelas:
            for agg in aggs:
                if (fvl == 'FVL_NUMERO_DIAS_ATRASO' and agg == 'SUM') or (fvl == 'FVL_QTD_TRANSACAO' and agg != 'SUM'):
                    continue
                else:
                    print(f'{agg}(CASE WHEN fbc_classificacao_pagamento = "{fbc}" AND {janela} = 1 THEN {fvl} END) as VAR_{count},')
                    count += 1

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_1,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_2,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_3,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_4,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_5,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_6,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_7,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_8,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1

In [150]:
book = spark.sql("""
                    select 
                        id_cliente,
                        ref,
                       SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_1,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_2,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_3,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_4,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_5,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_6,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_7,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_8,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_9,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_10,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_11,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_12,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_13,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_14,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_15,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_16,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_17,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_18,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_19,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_20,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_21,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_22,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_23,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_24,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_25,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_26,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_27,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_28,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_29,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_30,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_31,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_32,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_33,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_34,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_35,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_36,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_37,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_38,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_39,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_40,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_41,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_42,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_43,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_44,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_45,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_46,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_47,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_48,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_49,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_50,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_51,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_52,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_53,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_54,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_55,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_56,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_57,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_58,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_59,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_60,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_61,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_62,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_63,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_64,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_65,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_66,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_67,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_68,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_69,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_70,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_71,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO NO PRAZO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_72,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_73,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_74,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_75,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_76,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_77,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_78,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_79,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_80,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_81,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_82,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_83,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_84,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_85,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_86,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_87,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_88,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_89,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_90,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_91,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_92,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_93,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_94,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_95,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_96,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_97,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_98,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_99,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_100,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_101,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_102,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_103,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_104,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_105,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_106,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_107,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_108,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_109,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_110,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_111,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_112,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_113,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_114,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_115,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_116,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_117,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_118,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_119,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_120,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_121,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_122,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_123,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_124,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_125,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_126,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_127,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_128,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_129,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_130,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_131,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_132,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_133,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_134,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_135,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_136,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_137,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_138,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_139,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_140,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_141,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_142,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_143,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO ANTECIPADO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_144,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_145,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_146,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_147,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_148,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_149,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_150,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_151,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_152,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_153,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_154,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_155,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_156,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_157,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_158,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_159,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_160,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_161,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_162,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_163,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_164,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_165,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_166,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_167,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_168,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_169,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_170,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_171,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_172,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_173,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_174,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_175,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_176,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_177,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_178,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_179,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_180,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_181,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_182,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_183,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_184,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_185,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_186,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_187,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_188,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_189,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_190,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_191,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_192,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_193,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_194,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_195,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_196,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_197,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_198,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_199,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_200,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_201,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_202,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_203,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_204,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_205,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_206,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_207,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_208,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_209,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_210,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_211,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_212,
SUM(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_213,
AVG(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_214,
MIN(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_215,
MAX(CASE WHEN fbc_classificacao_pagamento = "SEM PAGAMENTO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_216,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_217,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_218,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_219,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_valor_fatura END) as VAR_220,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_221,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_222,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_223,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_valor_fatura END) as VAR_224,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_225,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_226,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_227,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_valor_fatura END) as VAR_228,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_229,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_230,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_231,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_valor_fatura END) as VAR_232,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_233,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_234,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_235,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_valor_fatura END) as VAR_236,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_237,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_238,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_239,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_valor_fatura END) as VAR_240,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_241,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_242,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_243,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_qtd_transacao END) as VAR_244,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_245,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_246,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_247,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_qtd_transacao END) as VAR_248,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_249,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_250,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_251,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_qtd_transacao END) as VAR_252,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_253,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_254,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_255,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_qtd_transacao END) as VAR_256,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_257,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_258,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_259,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_qtd_transacao END) as VAR_260,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_261,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_262,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_263,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_qtd_transacao END) as VAR_264,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_265,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_266,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_267,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND 1 = 1 THEN fvl_numero_dias_atraso END) as VAR_268,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_269,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_270,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_271,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U1M = 1 THEN fvl_numero_dias_atraso END) as VAR_272,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_273,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_274,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_275,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U3M = 1 THEN fvl_numero_dias_atraso END) as VAR_276,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_277,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_278,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_279,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U6M = 1 THEN fvl_numero_dias_atraso END) as VAR_280,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_281,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_282,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_283,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U9M = 1 THEN fvl_numero_dias_atraso END) as VAR_284,
SUM(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_285,
AVG(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_286,
MIN(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_287,
MAX(CASE WHEN fbc_classificacao_pagamento = "PAGAMENTO EM ATRASO" AND FLG_U12M = 1 THEN fvl_numero_dias_atraso END) as VAR_288
                   from janelas_tempo
                    group by 1,2
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
+----------+------+-------+-----------+-------+-------+-----+-----+-----+-----+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+--------+-----------+-------+-------+-------+----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+--------+-----------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+--------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+--------+-----------+-------+-------+--------+-----------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+------------------+-------+-------+-------+-------+-------+-------+-------+------------------+-------+-------+-------+------------------+-------+-------+-------+------------------+-------+-------+-------+------------------+-------+-------+
|id_cliente|   ref|  VAR_1|      VAR_2|  VAR_3|  VAR_4|VAR_5|VAR_6|VAR_7|VAR_8|  VAR_9|     VAR_10| VAR_11| VAR_12| VAR_13|     VAR_14| VAR_15| VAR_16| VAR_17|     VAR_18| VAR_19| VAR_20| VAR_21|     VAR_22| VAR_23| VAR_24|VAR_25|VAR_26|VAR_27|VAR_28|VAR_29|VAR_30|VAR_31|VAR_32|VAR_33|VAR_34|VAR_35|VAR_36|VAR_37|VAR_38|VAR_39|VAR_40|VAR_41|VAR_42|VAR_43|VAR_44|VAR_45|VAR_46|VAR_47|VAR_48|VAR_49|VAR_50|VAR_51|VAR_52|VAR_53|VAR_54|VAR_55|VAR_56|VAR_57|VAR_58|VAR_59|VAR_60|VAR_61|VAR_62|VAR_63|VAR_64|VAR_65|VAR_66|VAR_67|VAR_68|VAR_69|VAR_70|VAR_71|VAR_72|VAR_73|VAR_74|VAR_75|VAR_76|VAR_77|VAR_78|VAR_79|VAR_80|VAR_81|VAR_82|VAR_83|VAR_84|VAR_85|VAR_86|VAR_87|VAR_88|VAR_89|VAR_90|VAR_91|VAR_92|VAR_93|VAR_94|VAR_95|VAR_96|VAR_97|VAR_98|VAR_99|VAR_100|VAR_101|VAR_102|VAR_103|VAR_104|VAR_105|VAR_106|VAR_107|VAR_108|VAR_109|VAR_110|VAR_111|VAR_112|VAR_113|VAR_114|VAR_115|VAR_116|VAR_117|VAR_118|VAR_119|VAR_120|VAR_121|VAR_122|VAR_123|VAR_124|VAR_125|VAR_126|VAR_127|VAR_128|VAR_129|VAR_130|VAR_131|VAR_132|VAR_133|VAR_134|VAR_135|VAR_136|VAR_137|VAR_138|VAR_139|VAR_140|VAR_141|VAR_142|VAR_143|VAR_144| VAR_145|    VAR_146|VAR_147|VAR_148|VAR_149|   VAR_150|VAR_151|VAR_152|VAR_153|    VAR_154|VAR_155|VAR_156|VAR_157|    VAR_158|VAR_159|VAR_160|VAR_161|    VAR_162|VAR_163|VAR_164| VAR_165|    VAR_166|VAR_167|VAR_168|VAR_169|VAR_170|VAR_171|VAR_172|VAR_173|VAR_174|VAR_175|VAR_176|VAR_177|VAR_178|VAR_179|VAR_180|VAR_181|VAR_182|VAR_183|VAR_184|VAR_185|VAR_186|VAR_187|VAR_188|VAR_189|VAR_190|VAR_191|VAR_192|VAR_193|VAR_194|VAR_195|VAR_196|VAR_197|VAR_198|VAR_199|VAR_200|VAR_201|VAR_202|VAR_203|VAR_204|VAR_205|VAR_206|VAR_207|VAR_208|VAR_209|VAR_210|VAR_211|VAR_212|VAR_213|VAR_214|VAR_215|VAR_216| VAR_217|    VAR_218|VAR_219|VAR_220|VAR_221|    VAR_222|VAR_223|VAR_224|VAR_225|    VAR_226|VAR_227|VAR_228|VAR_229|    VAR_230|VAR_231|VAR_232| VAR_233|    VAR_234|VAR_235|VAR_236| VAR_237|    VAR_238|VAR_239|VAR_240|VAR_241|VAR_242|VAR_243|VAR_244|VAR_245|VAR_246|VAR_247|VAR_248|VAR_249|VAR_250|VAR_251|VAR_252|VAR_253|VAR_254|VAR_255|VAR_256|VAR_257|VAR_258|VAR_259|VAR_260|VAR_261|VAR_262|VAR_263|VAR_264|VAR_265|           VAR_266|VAR_267|VAR_268|VAR_269|VAR_270|VAR_271|VAR_272|VAR_273|           VAR_274|VAR_275|VAR_276|VAR_277|           VAR_278|VAR_279|VAR_280|VAR_281|           VAR_282|VAR_283|VAR_284|VAR_285|           VAR_286|VAR_287|VAR_288|
+----------+------+-------+-----------+-------+-------+-----+-----+-----+-----+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+--------+-----------+-------+-------+-------+----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+--------+-----------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+--------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+--------+-----------+-------+-------+--------+-----------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+------------------+-------+-------+-------+-------+-------+-------+-------+------------------+-------+-------+-------+------------------+-------+-------+-------+------------------+-------+-------+-------+------------------+-------+-------+
|       671|202409|   NULL|       NULL|   NULL|   NULL| NULL| NULL| NULL| NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 9865.37|2466.342500|1022.35|3806.66|   NULL|      NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|4829.01|2414.505000|1022.35|3806.66|7034.49|2344.830000|1022.35|3806.66| 9865.37|2466.342500|1022.35|3806.66|      4|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|      2|    1.0|      1|      1|      3|    1.0|      1|      1|      4|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|12683.81|2113.968333| 226.25|3791.20|1756.75|1756.750000|1756.75|1756.75|1983.00| 991.500000| 226.25|1756.75|1983.00| 991.500000| 226.25|1756.75| 5828.97|1457.242500| 226.25|3072.51| 9620.17|1924.034000| 226.25|3791.20|      6|    1.0|      1|      1|      1|    1.0|      1|      1|      2|    1.0|      1|      1|      2|    1.0|      1|      1|      4|    1.0|      1|      1|      5|    1.0|      1|      1|     43| 7.166666666666667|      2|     15|      2|    2.0|      2|      2|      8|               4.0|      2|      6|      8|               4.0|      2|      6|     25|              6.25|      2|     15|     29|               5.8|      2|     15|
|       319|202409|   NULL|       NULL|   NULL|   NULL| NULL| NULL| NULL| NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 4069.28|2034.640000| 192.09|3877.19|   NULL|      NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL| 192.09| 192.090000| 192.09| 192.09| 4069.28|2034.640000| 192.09|3877.19|      2|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|      1|    1.0|      1|      1|      2|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 5612.80|1870.933333|  67.10|4040.48|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|  67.10|  67.100000|  67.10|  67.10| 1572.32| 786.160000|  67.10|1505.22| 1572.32| 786.160000|  67.10|1505.22|      3|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|      1|    1.0|      1|      1|      2|    1.0|      1|      1|      2|    1.0|      1|      1|     24|               8.0|      3|     11|   NULL|   NULL|   NULL|   NULL|   NULL|              NULL|   NULL|   NULL|     10|              10.0|     10|     10|     13|               6.5|      3|     10|     13|               6.5|      3|     10|
|        60|202409|   NULL|       NULL|   NULL|   NULL| NULL| NULL| NULL| NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 9515.08|3171.693333| 859.77|4622.55| 859.77|859.770000| 859.77| 859.77| 859.77| 859.770000| 859.77| 859.77| 859.77| 859.770000| 859.77| 859.77| 859.77| 859.770000| 859.77| 859.77| 4892.53|2446.265000| 859.77|4032.76|      3|    1.0|      1|      1|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      2|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|11747.56|2936.890000| 640.40|4867.53|   NULL|       NULL|   NULL|   NULL|3773.79|3773.790000|3773.79|3773.79|3773.79|3773.790000|3773.79|3773.79|11747.56|2936.890000| 640.40|4867.53|11747.56|2936.890000| 640.40|4867.53|      4|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      4|    1.0|      1|      1|      4|    1.0|      1|      1|     45|             11.25|      4|     15|   NULL|   NULL|   NULL|   NULL|     11|              11.0|     11|     11|     11|              11.0|     11|     11|     45|             11.25|      4|     15|     45|             11.25|      4|     15|
|        99|202409|   NULL|       NULL|   NULL|   NULL| NULL| NULL| NULL| NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 5779.92|1926.640000| 365.89|4745.48| 668.55|668.550000| 668.55| 668.55| 668.55| 668.550000| 668.55| 668.55| 668.55| 668.550000| 668.55| 668.55| 668.55| 668.550000| 668.55| 668.55| 5414.03|2707.015000| 668.55|4745.48|      3|    1.0|      1|      1|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      2|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 8070.54|2017.635000| 275.09|3975.31|   NULL|       NULL|   NULL|   NULL| 799.01| 799.010000| 799.01| 799.01| 799.01| 799.010000| 799.01| 799.01| 7795.45|2598.483333| 799.01|3975.31| 8070.54|2017.635000| 275.09|3975.31|      4|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      3|    1.0|      1|      1|      4|    1.0|      1|      1|     29|              7.25|      4|     11|   NULL|   NULL|   NULL|   NULL|      4|               4.0|      4|      4|      4|               4.0|      4|      4|     18|               6.0|      4|      9|     29|              7.25|      4|     11|
|       626|202409|  83.17|  83.170000|  83.17|  83.17| NULL| NULL| NULL| NULL|   NULL|       NULL|   NULL|   NULL|  83.17|  83.170000|  83.17|  83.17|  83.17|  83.170000|  83.17|  83.17|  83.17|  83.170000|  83.17|  83.17|     1|   1.0|     1|     1|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|     1|   1.0|     1|     1|     1|   1.0|     1|     1|     1|   1.0|     1|     1|     0|   0.0|     0|     0|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|     0|   0.0|     0|     0|     0|   0.0|     0|     0|     0|   0.0|     0|     0|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|13671.33|2734.266000|1685.12|4705.87|   NULL|      NULL|   NULL|   NULL|1685.12|1685.120000|1685.12|1685.12|3850.88|1925.440000|1685.12|2165.76|8965.46|2241.365000|1685.12|2678.12|13671.33|2734.266000|1685.12|4705.87|      5|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|      1|    1.0|      1|      1|      2|    1.0|      1|      1|      4|    1.0|      1|      1|      5|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 3806.87|1268.956667| 228.05|2017.89|1560.93|1560.930000|1560.93|1560.93|1560.93|1560.930000|1560.93|1560.93|1560.93|1560.930000|1560.93|1560.93| 1560.93|1560.930000|1560.93|1560.93| 3578.82|1789.410000|1560.93|2017.89|      3|    1.0|      1|      1|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      2|    1.0|      1|      1|     13| 4.333333333333333|      1|      7|      1|    1.0|      1|      1|      1|               1.0|      1|      1|      1|               1.0|      1|      1|      1|               1.0|      1|      1|      8|               4.0|      1|      7|
|       220|202409|6638.13|3319.065000|1792.80|4845.33| NULL| NULL| NULL| NULL|1792.80|1792.800000|1792.80|1792.80|1792.80|1792.800000|1792.80|1792.80|1792.80|1792.800000|1792.80|1792.80|1792.80|1792.800000|1792.80|1792.80|     2|   1.0|     1|     1|  NULL|  NULL|  NULL|  NULL|     1|   1.0|     1|     1|     1|   1.0|     1|     1|     1|   1.0|     1|     1|     1|   1.0|     1|     1|     0|   0.0|     0|     0|  NULL|  NULL|  NULL|  NULL|     0|   0.0|     0|     0|     0|   0.0|     0|     0|     0|   0.0|     0|     0|     0|   0.0|     0|     0|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 7277.93|2425.976667|1701.51|2911.28|   NULL|      NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|2911.28|2911.280000|2911.28|2911.28|5576.42|2788.210000|2665.14|2911.28| 7277.93|2425.976667|1701.51|2911.28|      3|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|      1|    1.0|      1|      1|      2|    1.0|      1|      1|      3|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 4472.66|4472.660000|4472.66|4472.66|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL| 4472.66|4472.660000|4472.66|4472.66| 4472.66|4472.660000|4472.66|4472.66|      1|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      3|               3.0|      3|      3|   NULL|   NULL|   NULL|   NULL|   NULL|              NULL|   NULL|   NULL|   NULL|              NULL|   NULL|   NULL|      3|               3.0|      3|      3|      3|               3.0|      3|      3|
|        22|202409|   NULL|       NULL|   NULL|   NULL| NULL| NULL| NULL| NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 9008.06|3002.686667|2814.88|3299.52|   NULL|      NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|6193.18|3096.590000|2893.66|3299.52|6193.18|3096.590000|2893.66|3299.52| 9008.06|3002.686667|2814.88|3299.52|      3|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|      2|    1.0|      1|      1|      2|    1.0|      1|      1|      3|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 9077.45|3025.816667|2024.79|4800.25|2024.79|2024.790000|2024.79|2024.79|2024.79|2024.790000|2024.79|2024.79|2024.79|2024.790000|2024.79|2024.79| 9077.45|3025.816667|2024.79|4800.25| 9077.45|3025.816667|2024.79|4800.25|      3|    1.0|      1|      1|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      1|    1.0|      1|      1|      3|    1.0|      1|      1|      3|    1.0|      1|      1|     28| 9.333333333333334|      8|     12|      8|    8.0|      8|      8|      8|               8.0|      8|      8|      8|               8.0|      8|      8|     28| 9.333333333333334|      8|     12|     28| 9.333333333333334|      8|     12|
|       458|202409|   NULL|       NULL|   NULL|   NULL| NULL| NULL| NULL| NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|15390.07|3078.014000|1576.26|4647.61|   NULL|      NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|7715.33|3857.665000|3067.72|4647.61|13709.66|3427.415000|1576.26|4647.61|      5|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|      2|    1.0|      1|      1|      4|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 5342.04|1780.680000| 755.69|3695.09|3695.09|3695.090000|3695.09|3695.09|4586.35|2293.175000| 891.26|3695.09|5342.04|1780.680000| 755.69|3695.09| 5342.04|1780.680000| 755.69|3695.09| 5342.04|1780.680000| 755.69|3695.09|      3|    1.0|      1|      1|      1|    1.0|      1|      1|      2|    1.0|      1|      1|      3|    1.0|      1|      1|      3|    1.0|      1|      1|      3|    1.0|      1|      1|     29| 9.666666666666666|      6|     13|      6|    6.0|      6|      6|     19|               9.5|      6|     13|     29| 9.666666666666666|      6|     13|     29| 9.666666666666666|      6|     13|     29| 9.666666666666666|      6|     13|
|       812|202409|   NULL|       NULL|   NULL|   NULL| NULL| NULL| NULL| NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|14307.34|2861.468000| 202.01|4750.55|   NULL|      NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL| 202.01| 202.010000| 202.01| 202.01|9098.92|3032.973333| 202.01|4750.55|10842.80|2710.700000| 202.01|4750.55|      5|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|      1|    1.0|      1|      1|      3|    1.0|      1|      1|      4|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 4497.59|1499.196667|1124.57|1894.95|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL| 1124.57|1124.570000|1124.57|1124.57| 4497.59|1499.196667|1124.57|1894.95|      3|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|      1|    1.0|      1|      1|      3|    1.0|      1|      1|     35|11.666666666666666|      8|     14|   NULL|   NULL|   NULL|   NULL|   NULL|              NULL|   NULL|   NULL|   NULL|              NULL|   NULL|   NULL|      8|               8.0|      8|      8|     35|11.666666666666666|      8|     14|
|       664|202409|   NULL|       NULL|   NULL|   NULL| NULL| NULL| NULL| NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 8236.18|2059.045000|1081.56|2854.35|   NULL|      NULL|   NULL|   NULL|   NULL|       NULL|   NULL|   NULL|5711.44|1903.813333|1081.56|2854.35|5711.44|1903.813333|1081.56|2854.35| 8236.18|2059.045000|1081.56|2854.35|      4|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|      3|    1.0|      1|      1|      3|    1.0|      1|      1|      4|    1.0|      1|      1|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL|   NULL| 9539.34|1907.868000| 863.88|3849.90| 863.88| 863.880000| 863.88| 863.88|4168.92|1389.640000| 863.88|1819.01|4168.92|1389.640000| 863.88|1819.01| 4168.92|1389.640000| 863.88|1819.01| 5689.44|1422.360000| 863.88|1819.01|      5|    1.0|      1|      1|      1|    1.0|      1|      1|      3|    1.0|      1|      1|      3|    1.0|      1|      1|      3|    1.0|      1|      1|      4|    1.0|      1|      1|     65|              13.0|     11|     15|     11|   11.0|     11|     11|     37|12.333333333333334|     11|     14|     37|12.333333333333334|     11|     14|     37|12.333333333333334|     11|     14|     50|              12.5|     11|     14|
+----------+------+-------+-----------+-------+-------+-----+-----+-----+-----+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+--------+-----------+-------+-------+-------+----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+--------+-----------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+--------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+-------+-----------+-------+-------+--------+-----------+-------+-------+--------+-----------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+------------------+-------+-------+-------+-------+-------+-------+-------+------------------+-------+-------+-------+------------------+-------+-------+-------+------------------+-------+-------+-------+------------------+-------+-------+

In [153]:
book.write.mode("append").partitionBy("ref").parquet("s3://data-lake-elt-303004020922-us-east-1-an/03_refined/book/book_fatura")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…